# Day 2
LangChain Tools, Chains, Memory & Your First Framework Agent


### Task 1: LangChain Setup & Core Concepts

#### Mapping Day 1 (raw Python) to LangChain

| Day 1 raw-Python piece | LangChain equivalent |
|---|---|
| client = Groq(...) + manual chat.completions.create() calls | ChatGroq(model=...)  an **LLM wrapper** that standardizes .invoke() / .stream() across providers |
| Hand-written TOOLS JSON schema list + TOOL_REGISTRY dict | @tool decorator  turns a plain Python function into a schema-carrying Tool object automatically, using the function signature + docstring |
| run_agent()'s for loop (call LLM → check tool_calls → execute → append tool_result → repeat) | create_tool_calling_agent() + AgentExecutor  the loop itself is now hidden inside AgentExecutor.invoke() |
| AgentState.messages (conversation history you manually appended to) | ConversationBufferMemory / RunnableWithMessageHistory  history is tracked and re-injected for you |
| AgentState.state (your own scratchpad: iterations, tools_used, observations) | agent_scratchpad placeholder inside the prompt  LangChain manages *its* scratchpad, but your custom working-memory fields (loop-risk counters, etc.) still have to be built yourself if you want them |
| safe_api_call() try/except wrapper | AgentExecutor(..., handle_parsing_errors=True) + per-tool handle_tool_error=True |


In [31]:
# Below is the new method now

# from langchain.agents import create_agent
# REMOVE THIS OLD BLOCK:
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful assistant."),
#     ("human", "{input}"),
#     ("placeholder", "{agent_scratchpad}"),
# ])
# agent = create_tool_calling_agent(llm, tools, prompt)
# agent_executor = AgentExecutor(agent=agent, tools=tools)

# REPLACE WITH THIS NEW BLOCK:
# agent = create_agent(
#     model=llm, 
#     tools=tools, 
#     system_prompt="You are a helpful assistant."
# )
# REMOVE THIS OLD CALL:
# response = agent_executor.invoke({"input": "What is 432 multiplied by 12?"})
# print(response["output"])

# REPLACE WITH THIS NEW CALL:
# response = agent.invoke({
#     "messages": [
#         {"role": "user", "content": "What is 432 multiplied by 12?"}
#     ]
# })

# # The final answer is the content of the very last message in the response
# print(response["messages"][-1].content)


In [32]:
import json
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor



load_dotenv()

MODEL = "llama-3.3-70b-versatile"

llm = ChatGroq(model=MODEL, temperature=0)

### A basic LCEL pipeline

LLMChain is the old way to do this, the modern  way is **LCEL** (LangChain
Expression Language), which uses the | pipe operator to compose Runnable objects.


In [33]:
basic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant."),
    ("human", "{question}"),
])

basic_chain = basic_prompt | llm | StrOutputParser()

print(basic_chain.invoke({"question": "In one sentence, what is a ReAct agent?"}))

A ReAct agent is a type of antibiotic resistance-breaking compound that restores the effectiveness of certain antibiotics against resistant bacteria.


## Under the Hood: The | Operator in LangChain

* The Runnable Interface: All LangChain components (prompts, models, parsers) share a standard interface with methods like .invoke(), .batch(), and .stream().
* Syntactic Sugar: The | operator overloads Python's __or__ method to combine these components into a single RunnableSequence.
* Data Flow: Chaining a | b | c is simply function composition—the output of a is fed directly into b, and b's output goes into c.
* Lazy Execution: The operator just builds the pipeline. No actual processing or network calls happen until .invoke() is explicitly called on the final chain.

## Task 2: Define & Register Tools

* The Tools: We are registering three tools: calculator and weather_lookup (reused), plus a new get_product_price tool that queries an external products.json database.
* The Power of @tool: This decorator automatically converts a function's type hints into a JSON schema and its docstring into the tool's description within the system prompt.
* Why Docstrings are Critical: The model cannot read your source code. The docstring and type hints are the *only* information it uses to decide when and how to call a tool. A vague description leads to missed or incorrect calls, while a precise docstring ensures reliable tool selection.

In [34]:
# Build a tiny local JSON "database" so get_product_price reads real external data,
PRODUCTS_DB_PATH = "products.json"

products_seed = {
    "laptop_a": {"name": "Laptop A (Budget)", "price_usd": 550, "category": "laptop"},
    "laptop_b": {"name": "Laptop B (Mid-range)", "price_usd": 950, "category": "laptop"},
    "laptop_c": {"name": "Laptop C (Premium)", "price_usd": 1800, "category": "laptop"},
    "monitor_a": {"name": "Monitor A (24-inch)", "price_usd": 180, "category": "monitor"},
    "monitor_b": {"name": "Monitor B (27-inch 4K)", "price_usd": 420, "category": "monitor"},
}

with open(PRODUCTS_DB_PATH, "w") as f:
    json.dump(products_seed, f, indent=2)

print(f"Wrote {PRODUCTS_DB_PATH} with {len(products_seed)} products.")

Wrote products.json with 5 products.


In [35]:
@tool
def calculator(a: float, b: float, operation: str) -> str:
    """Perform basic arithmetic. operation must be one of:
    'add', 'subtract', 'multiply', 'divide'. Use this whenever the user
    asks you to compute, compare numerically, or do math on two numbers."""
    try:
        if operation == "add":
            result = a + b
        elif operation == "subtract":
            result = a - b
        elif operation == "multiply":
            result = a * b
        elif operation == "divide":
            if b == 0:
                return json.dumps({"success": False, "error": "Cannot divide by zero."})
            result = a / b
        else:
            return json.dumps({"success": False, "error": f"Unknown operation: {operation}"})
        return json.dumps({"success": True, "result": result})
    except Exception as e:
        return json.dumps({"success": False, "error": str(e)})


WEATHER_DATA = {
    "lahore": {"temperature": 32, "condition": "Sunny"},
    "islamabad": {"temperature": 28, "condition": "Partly cloudy"},
    "karachi": {"temperature": 31, "condition": "Humid"},
    "faisalabad": {"temperature": 33, "condition": "Sunny"},
}


@tool
def weather_lookup(city: str) -> str:
    """Look up current weather (temperature in Celsius, condition) for a named
    city. Only works for Lahore, Islamabad, Karachi, and Faisalabad. Use this
    when the user asks about weather or temperature in a specific city."""
    city_key = city.strip().lower()
    if city_key not in WEATHER_DATA:
        return json.dumps({"success": False, "error": f"Weather data not available for '{city}'."})
    weather = WEATHER_DATA[city_key]
    return json.dumps({"success": True, "city": city, **weather})


@tool
def get_product_price(product_id: str) -> str:
    """Look up the price and category of a product from the product catalog.
    product_id must be a lowercase, underscore-separated id such as
    'laptop_a', 'laptop_b', 'monitor_a'. Use this whenever the user asks for
    the price of a product or wants to compare two products' prices. If you
    don't know the exact product_id, ask the user or try the closest match."""
    with open(PRODUCTS_DB_PATH) as f:
        db = json.load(f)
    if product_id not in db:
        raise ValueError(
            f"'{product_id}' not found in catalog. Available ids: {list(db.keys())}"
        )
    return json.dumps({"success": True, "product_id": product_id, **db[product_id]})



tools = [calculator, weather_lookup, get_product_price]
for t in tools:
    print(f"{t.name}: {t.description[:70]}...")

calculator: Perform basic arithmetic. operation must be one of:
'add', 'subtract',...
weather_lookup: Look up current weather (temperature in Celsius, condition) for a name...
get_product_price: Look up the price and category of a product from the product catalog.
...


#### Task 3: Build an Agent with create_tool_calling_agent + AgentExecutor

verbose=True prints LangChain's own reasoning trace (tool calls + observations) as the
agent runs. We capture one multi-step run below and annotate it afterward.


In [36]:
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools. "
               "Use tools whenever they would give a more accurate answer "
               "than guessing. Show your reasoning briefly before acting."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
)

trace_result = agent_executor.invoke({
    "input": (
        "Look up the weather in Lahore and Islamabad, and tell me which "
        "city is warmer and by how many degrees."
    )
})

print("\nFINAL OUTPUT:")
print(trace_result["output"])



> Entering new AgentExecutor chain...

Invoking: `weather_lookup` with `{'city': 'Lahore'}`
responded: To determine which city is warmer and by how many degrees, I need to look up the current weather in Lahore and Islamabad. I will use the weather_lookup function to get the current temperature in each city.



{"success": true, "city": "Lahore", "temperature": 32, "condition": "Sunny"}
Invoking: `weather_lookup` with `{'city': 'Islamabad'}`
responded: To determine which city is warmer and by how many degrees, I need to look up the current weather in Lahore and Islamabad. I will use the weather_lookup function to get the current temperature in each city.



{"success": true, "city": "Islamabad", "temperature": 28, "condition": "Partly cloudy"}Lahore is warmer than Islamabad by 4 degrees.

> Finished chain.

FINAL OUTPUT:
Lahore is warmer than Islamabad by 4 degrees.


## Annotated Trace Breakdown (Task 3)

The verbose=True output reveals the following sequence:

1. Reason: The model determines it needs external data rather than relying on its internal memory to make a comparison.
2. Act: It triggers a weather_lookup tool call for "Lahore".
3. Observe: AgentExecutor runs the tool behind the scenes and returns the JSON result as an observation.
4. Repeat: The Reason → Act → Observe cycle triggers again for "Islamabad".
5. Final Response: With both data points in context, the model calculates the difference (32°C − 28°C = 4°C) and generates the final natural-language output (trace_result["output"]).

---

## AgentExecutor vs. Manual "Day 1" Loop

While the core Reason-Act-Observe pattern remains identical, AgentExecutor abstracts away several mechanics that were previously handled manually:

* Hidden API Payloads: The exact JSON requests and responses sent to the model (e.g., raw tool_calls) are now obscured behind LangChain's formatting.
* Automated Context Tracking: The message history (agent_scratchpad) is constructed automatically from action/observation pairs, replacing manual list appends.
* Simplistic Loop Protection: AgentExecutor relies on a basic max_iterations limit. It lacks the nuanced, custom infinite-loop protections (like checking for repeated identical calls) manually built in previous iterations.

## Task 4: Add Memory


In [37]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# In-memory per-session chat history store
_session_store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in _session_store:
        _session_store[session_id] = InMemoryChatMessageHistory()
    return _session_store[session_id]

agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

session_config = {"configurable": {"session_id": "budget-client-1"}}

c:\Users\MEE\Netixsol_intern_projects\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [38]:
# Turn 1: "Find the price of X"
turn1 = agent_with_memory.invoke(
    {"input": "What is the price of laptop_a?"},
    config=session_config,
)
print("TURN 1:", turn1["output"])



> Entering new AgentExecutor chain...

Invoking: `get_product_price` with `{'product_id': 'laptop_a'}`
responded: To find the price of laptop_a, I should look up the product in the catalog. 


{"success": true, "product_id": "laptop_a", "name": "Laptop A (Budget)", "price_usd": 550, "category": "laptop"}The price of laptop_a is $550.

> Finished chain.
TURN 1: The price of laptop_a is $550.


In [39]:
# Turn 2: "Now compare it to Y"
turn2 = agent_with_memory.invoke(
    {"input": "Now compare it to laptop_b."},
    config=session_config,
)
print("TURN 2:", turn2["output"])



> Entering new AgentExecutor chain...

Invoking: `get_product_price` with `{'product_id': 'laptop_b'}`
responded: To compare the prices, I need to look up the price of laptop_b. 



{"success": true, "product_id": "laptop_b", "name": "Laptop B (Mid-range)", "price_usd": 950, "category": "laptop"}
Invoking: `get_product_price` with `{'product_id': 'laptop_a'}`


{"success": true, "product_id": "laptop_a", "name": "Laptop A (Budget)", "price_usd": 550, "category": "laptop"}
Invoking: `calculator` with `{'a': 950, 'b': 550, 'operation': 'subtract'}`
responded: To compare the prices of laptop_a and laptop_b, we need to perform a subtraction operation. 


{"success": true, "result": 400.0}The price of laptop_a is $550 and the price of laptop_b is $950. The difference between the two prices is $400.

> Finished chain.
TURN 2: The price of laptop_a is $550 and the price of laptop_b is $950. The difference between the two prices is $400.


In [40]:
# Turn 3: "Which one should I recommend to a budget-conscious client?"
turn3 = agent_with_memory.invoke(
    {"input": "Which one should I recommend to a budget-conscious client?"},
    config=session_config,
)
print("TURN 3:", turn3["output"])



> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'a': 550, 'b': 950, 'operation': 'subtract'}`
responded: To determine which laptop is more budget-friendly, I need to compare their prices. 



{"success": true, "result": -400.0}Since laptop_a is $400 cheaper than laptop_b, I would recommend laptop_a to a budget-conscious client.

> Finished chain.
TURN 3: Since laptop_a is $400 cheaper than laptop_b, I would recommend laptop_a to a budget-conscious client.


### Task 5: Structured Output & Error Handling

Force the agent's final recommendation into a Pydantic schema. Since AgentExecutor's
own output is free-text, the clean pattern is: run the agent to gather facts, then pass
its answer through a second LLM call bound to with_structured_output().


In [41]:
from pydantic import BaseModel, Field

class ProductRecommendation(BaseModel):
    """A structured recommendation for client."""
    recommended_product_id: str = Field(description="The product_id being recommended, e.g. 'laptop_a'")
    price_usd: float = Field(description="The price of the recommended product in USD")
    reason: str = Field(description="A short reason for the recommendation")
    budget_friendly: bool = Field(description="Whether this recommendation prioritizes budget over features")


structured_llm = llm.with_structured_output(ProductRecommendation)

structured_prompt = (
    "Based on this analysis, produce a structured recommendation.\n\n"
    f"Analysis: {turn3['output']}"
)

structured_result = structured_llm.invoke(structured_prompt)
print(structured_result)
print(type(structured_result))

recommended_product_id='laptop_a' price_usd=400.0 reason='laptop_a is $400 cheaper than laptop_b' budget_friendly=True
<class '__main__.ProductRecommendation'>


#### Error handling for a pseudo tool

A tool that throws an exception 50% of the time, to observe recovery behavior.


In [42]:
import random
from langchain_core.tools import ToolException

@tool
def flaky_price_lookup(product_id: str) -> str:
    """Look up a product's price from a slow, occasionally-unreliable external
    price feed (simulates a real API that sometimes times out). Prefer
    get_product_price for reliable lookups; only use this tool if the user
    specifically asks to check the 'live' or 'external' price feed."""
    if random.random() < 0.5:
        raise ToolException("Simulated timeout contacting external price feed.")
    with open(PRODUCTS_DB_PATH) as f:
        db = json.load(f)
    if product_id not in db:
        raise ToolException(f"'{product_id}' not found.")
    return json.dumps({"success": True, "product_id": product_id, **db[product_id]})

flaky_price_lookup.handle_tool_error = True


flaky_tools = tools + [flaky_price_lookup]
flaky_agent = create_tool_calling_agent(llm, flaky_tools, agent_prompt)
flaky_executor = AgentExecutor(
    agent=flaky_agent,
    tools=flaky_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=6,
)

flaky_result = flaky_executor.invoke({
    "input": "Check the live external price feed for laptop_c and tell me the price."
})
print("\nFINAL OUTPUT:")
print(flaky_result["output"])



> Entering new AgentExecutor chain...

Invoking: `flaky_price_lookup` with `{'product_id': 'laptop_c'}`
responded: To get the live external price of 'laptop_c', I should use the function that looks up a product's price from a slow, occasionally-unreliable external price feed. 



Simulated timeout contacting external price feed.
Invoking: `flaky_price_lookup` with `{'product_id': 'laptop_c'}`


Simulated timeout contacting external price feed.
Invoking: `flaky_price_lookup` with `{'product_id': 'laptop_c'}`


Simulated timeout contacting external price feed.
Invoking: `get_product_price` with `{'product_id': 'laptop_c'}`
responded: I'm having trouble getting the price from the external price feed. Let me try to get the price from the product catalog instead.


{"success": true, "product_id": "laptop_c", "name": "Laptop C (Premium)", "price_usd": 1800, "category": "laptop"}The price of laptop_c from the product catalog is $1800.

> Finished chain.

FINAL OUTPUT:
The price of laptop_c 

#### Graceful Error Handling

* Preventing crashes (handle_tool_error=True): Applying this setting to a tool means LangChain catches an exception raised inside it instead of letting it propagate and crash AgentExecutor.invoke(). The catch mechanism only intercepts ToolException (from langchain_core.tools)  a raw ConnectionError, ValueError, etc. is *not* caught by this flag and will still crash the run, so the tool has to explicitly raise ToolException for the flag to do anything.
* Enabling self-correction: When a caught ToolException fires, LangChain converts it into a text observation and feeds it back to the model as if it were a normal tool result. This lets the agent dynamically retry the failed tool or switch to an alternative (e.g. falling back from flaky_price_lookup to get_product_price) instead of the whole run halting.

#### LangChain Abstractions: Pros & Cons

What LangChain Simplifies:
It eliminates manual boilerplate. Components like @tool, AgentExecutor, and RunnableWithMessageHistory automatically manage JSON schemas, message serialization, and conversation memory.

Where the "Magic" Leaks (Hidden Drawbacks):

* Obscured Prompts: The exact prompt structure assembled behind agent_scratchpad is hidden unless you use external monitoring tools like LangSmith.
* Blunt Safeguards: It relies on a simple max_iterations limit rather than the nuanced, custom infinite-loop detection you can build manually.
* Opaque Fallbacks: The with_structured_output() method silently switches between tool-calling and JSON mode depending on the model provider, which can complicate debugging if a provider lacks support for the chosen mechanism.